<a href="https://colab.research.google.com/github/ibarr123/BUS1182026/blob/main/PR2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langgraph langchain langchain-openai python-dotenv pypdf pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 21.3 MB/s eta 0:00:00


In [2]:
import os
import re
import pandas as pd
from pypdf import PdfReader
from typing import Literal, TypedDict
from typing_extensions import Annotated

from google.colab import userdata

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
# Load API key from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

llm = ChatOpenAI(model="gpt-5-nano")

# File paths
SPEC_PDF = "/content/EcoSprint_Specification_Document.pdf"
PRODUCTS_PDF = "/content/Laptop product descriptions.pdf"
PRICING_CSV = "/content/Laptop pricing.csv"
ORDERS_CSV = "/content/Laptop Orders.csv"

In [4]:
def read_pdf_text(path):
    text_parts = []
    reader = PdfReader(path)
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text_parts.append(page_text)
    return "\n".join(text_parts)

spec_text = read_pdf_text(SPEC_PDF)
product_desc_text = read_pdf_text(PRODUCTS_PDF)

pricing_df = pd.read_csv(PRICING_CSV)
orders_df = pd.read_csv(ORDERS_CSV)

print("Pricing columns:", pricing_df.columns.tolist())
print("Orders columns:", orders_df.columns.tolist())

Pricing columns: ['Name', 'Price', 'ShippingDays']
Orders columns: ['Order ID', 'Product Ordered', 'Quantity Ordered', 'Delivery Date']


In [5]:
def chunk_text(text, chunk_size=1200, overlap=200):
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

spec_chunks = chunk_text(spec_text)
product_chunks = chunk_text(product_desc_text)

In [6]:
def keyword_score(query, text):
    words = re.findall(r"\w+", query.lower())
    text_lower = text.lower()
    score = 0
    for word in words:
        if len(word) > 2:
            score += text_lower.count(word)
    return score

def retrieve_pdf_context(user_input, top_k=3):
    scored = []

    for i, chunk in enumerate(spec_chunks):
        score = keyword_score(user_input, chunk)
        if score > 0:
            scored.append((score, f"EcoSprint_Specification_Document.pdf | chunk {i}", chunk))

    for i, chunk in enumerate(product_chunks):
        score = keyword_score(user_input, chunk)
        if score > 0:
            scored.append((score, f"Laptop product descriptions.pdf | chunk {i}", chunk))

    scored.sort(key=lambda x: x[0], reverse=True)
    top = scored[:top_k]

    if not top:
        return "No relevant PDF context found."

    return "\n\n".join([f"SOURCE: {source}\n{chunk}" for _, source, chunk in top])

In [7]:
def find_order_row(user_input):
    text = user_input.lower()
    order_match = re.search(r"\b\d{3,}\b", text)
    if not order_match:
        return None

    order_id = order_match.group(0)

    for col in orders_df.columns:
        matches = orders_df[orders_df[col].astype(str).str.lower() == order_id.lower()]
        if not matches.empty:
            return matches.iloc[0].to_dict()

    return None

def format_order_info(order_row):
    if not order_row:
        return "No matching order found in Laptop Orders.csv."
    return "\n".join([f"{k}: {v}" for k, v in order_row.items()])

In [8]:
def find_pricing_rows(user_input, top_k=5):
    text = user_input.lower()
    results = []

    for _, row in pricing_df.iterrows():
        row_text = " ".join([str(x) for x in row.values]).lower()
        score = keyword_score(text, row_text)
        if score > 0:
            results.append((score, row.to_dict()))

    results.sort(key=lambda x: x[0], reverse=True)
    return [row for _, row in results[:top_k]]

def format_pricing_rows(rows):
    if not rows:
        return "No matching products found in Laptop pricing.csv."

    blocks = []
    for i, row in enumerate(rows, start=1):
        lines = [f"Match {i}:"]
        for k, v in row.items():
            lines.append(f"{k}: {v}")
        blocks.append("\n".join(lines))
    return "\n\n".join(blocks)

In [9]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]
    route: str

In [10]:
def router_node(state: ChatState):
    system = SystemMessage(
        content=(
            "Classify the user's latest request as exactly one of: "
            "order_status, refund_policy, product_recommendation. "
            "Use refund_policy for returns, refunds, warranty, or policy questions. "
            "Only return the label."
        )
    )

    user_msg = state["messages"][-1]
    response = llm.invoke([system, user_msg])
    route = response.content.strip().lower()

    if route not in ["order_status", "refund_policy", "product_recommendation"]:
        route = "product_recommendation"

    return {"route": route}

In [11]:
def order_node(state: ChatState):
    text = state["messages"][-1].content
    order_info = format_order_info(find_order_row(text))

    system = SystemMessage(
        content=(
            "You are a customer support assistant. "
            "Answer only using the order information provided below. "
            "If no matching order was found, ask the user for a valid order number.\n\n"
            f"{order_info}"
        )
    )

    response = llm.invoke([system] + state["messages"])
    return {"messages": [AIMessage(content=response.content)]}

In [19]:
def refund_node(state: ChatState):
    fallback_refund_policy = """
EcoSprint Refund Policy:
- Returns accepted within 30 days of delivery
- Item must be unused and in original packaging for full refund
- Opened items may be eligible for store credit depending on condition
- Final sale items cannot be returned
"""

    system = SystemMessage(
        content=(
            "You are a customer support agent. "
            "Answer refund, return, warranty, or policy questions using only the refund policy below. "
            "Do not invent any additional policy details. "
            "If the user asks something outside this policy, say that only the listed refund policy information is available.\n\n"
            f"{fallback_refund_policy}"
        )
    )

    response = llm.invoke([system] + state["messages"])
    return {"messages": [AIMessage(content=response.content)]}

In [20]:
def product_node(state: ChatState):
    text = state["messages"][-1].content
    pricing_info = format_pricing_rows(find_pricing_rows(text, top_k=5))
    pdf_context = retrieve_pdf_context(text, top_k=3)

    system = SystemMessage(
        content=(
            "You are a sales assistant. "
            "Recommend products only from the pricing data and product-description context below. "
            "Do not invent products, prices, or specs. "
            "If the user asks for something not found, say so clearly.\n\n"
            "PRICING DATA:\n"
            f"{pricing_info}\n\n"
            "PRODUCT DESCRIPTION CONTEXT:\n"
            f"{pdf_context}"
        )
    )

    response = llm.invoke([system] + state["messages"])
    return {"messages": [AIMessage(content=response.content)]}

In [21]:
def route_decision(state: ChatState) -> Literal["order_status", "refund_policy", "product_recommendation"]:
    return state["route"]

builder = StateGraph(ChatState)

builder.add_node("router", router_node)
builder.add_node("order_status", order_node)
builder.add_node("refund_policy", refund_node)
builder.add_node("product_recommendation", product_node)

builder.add_edge(START, "router")
builder.add_conditional_edges(
    "router",
    route_decision,
    {
        "order_status": "order_status",
        "refund_policy": "refund_policy",
        "product_recommendation": "product_recommendation",
    },
)

builder.add_edge("order_status", END)
builder.add_edge("refund_policy", END)
builder.add_edge("product_recommendation", END)

memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

In [ ]:
def chat():
    print("EcoSprint Laptop Assistant")
    print("Type 'quit' to exit.")
    config = {"configurable": {"thread_id": "user-1"}}

    while True:
        user_input = input("You: ")
        if user_input.lower() == "quit":
            break

        result = graph.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config=config
        )

        print("Bot:", result["messages"][-1].content)

chat()

EcoSprint Laptop Assistant
Type 'quit' to exit.
You: i need a laptop for school make a recommendation
Bot: Recommendation: NanoEdge Flex

- Why: It’s a versatile 2-in-1 laptop with an Apple M1 Pro chip, 16GB unified memory, and a 512GB SSD. The 2-in-1 design is great for taking notes in tablet mode, giving you flexibility for classes, lectures, and presentations while staying portable and efficient for everyday school tasks.

Alternative options if you want more power or storage:
- AlphaBook Pro: 12th Gen Intel i7, 16GB RAM, 1TB SSD — strong performance in a slim ultrabook.
- GammaAir X: Ryzen 7, 32GB RAM, 512GB NVMe — excellent multitasking headroom if you run many apps at once.

Would you prefer a 2-in-1 style or a traditional clamshell, and is extra RAM or more storage your priority?
You: extra ram
Bot: Nice—focusing on extra RAM. Here are the RAM-rich options from the descriptions:

- SpectraBook S: 64GB RAM, 2TB SSD, Intel Core i9. Best for heavy multitasking, video editing, or ot